# Figure 5 b–e — two-colour DNA-PAINT of a BSC-1 cell, ATTO 655 + Cy3B (Ximea)

**Makes up:** Figure 5 b (two-colour render), c (zoom with line profiles), d (mitochondria zoom), e (FRC of each channel).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/MassiveCells/250pMATTO655_750pMCy3B_20perc561_60mW638_1/` — raw Ximea TIFFs
(20 files, ~77 GB) plus the acquisition's MicroManager metadata. The deposition holds raw
data only, so this notebook fits it first.

MASSIVE Photonics two-colour standard: vimentin labelled with ATTO 655, mitochondria with
Cy3B, imaged in one acquisition and separated by their spectral fingerprints.

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits only the first TIFF (~4 GB), enough to run the whole
pipeline end to end, with far fewer localisations than the published figure. `FAST_MODE = False`
downloads and fits the full acquisition (~77 GB; many hours of fitting; run in `tmux`/`screen`).
Hand-picked regions are in camera pixels of the full acquisition; in `FAST_MODE` they may be sparse.

---
## 0 — Imports

In [ ]:
import json
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colors as mcolors
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from skimage.draw import line as bresenham_line

# _biostudies_download.py lives beside this notebook (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve()))
import _biostudies_download as bd

from pyS3M import IOFunctions
from pyS3M import HelperFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SpectralFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase
from pyS3M import render
from pyS3M import FiducialDetection
from pyS3M import SM_extractionfunctions
from pyS3M.FRCFunctions import fire
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions
from pyS3M.LinkingFunctions import link_localisations

IO = IOFunctions.IO_Functions()
H_F = HelperFunctions.Helper_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
plotter = PlottingBase.PublicationPlotter(dark_background=False)

FD = FiducialDetection.FiducialDetector()
SM_E = SM_extractionfunctions.extract_SMs()

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: first TIFF only (~4 GB, illustrative). False: all 20 TIFFs (~77 GB, the published data).

PIXEL_SIZE_NM = 69.0   # Ximea camera pixel

REPO_ROOT = Path.cwd().resolve().parents[1]   # notebooks/figures -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "Figure_5b-e_MassiveCells"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_DIR at a drive with room for the download. Keep one folder per acquisition and
# per FAST_MODE setting: the fit reads every TIFF inside it.
DATA_DIR  = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

gain_map   = IO.read_tiff(CALIB_DIR / "gain.tif")
offset_map = IO.read_tiff(CALIB_DIR / "offset.tif")
variance   = IO.read_tiff(CALIB_DIR / "variance.tif")
read_noise = IO.read_tiff(CALIB_DIR / "readnoise.tif")
rqe        = IO.read_tiff(CALIB_DIR / "rqe.tif")

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download the raw data from BioStudies (`S-BIAD3210`)

Downloads the first TIFF (`FAST_MODE`) or the whole series, plus the acquisition's MicroManager
`*_metadata.txt` in both cases: it holds the camera ROI the frames were cropped to, which the fit
needs to crop the calibration maps. Re-running is safe: complete files are skipped and interrupted
ones resume.

In [ ]:
REMOTE_FOLDER = "2026_Multicolour_Paper/Ximea/MassiveCells/250pMATTO655_750pMCy3B_20perc561_60mW638_1"

downloaded = bd.download_acquisition(REMOTE_FOLDER, DATA_DIR, first_file_only=FAST_MODE)

# Camera ROI of the acquisition (the fitted xc/yc are relative to its corner).
roi_x, roi_y, width, height = IO.metadata_reader_imageJ(downloaded["metadata"][0])
print(f"ROI: x={roi_x} y={roi_y}, {width} x {height} px")

---
## 3 — Spot detection and fitting

The paper's settings: pfa 1e-3, sigma 1.5, fraction_true 0.2, peak wavelength 0.65 um, 12 px ROI.
`fit_imaging_data` treats the TIFFs as one series and writes `Localisations.h5` into `DATA_DIR`.
This takes about 1 h in FAST_MODE and many hours in full. A record of what was fitted is saved beside it, so re-running the notebook
reuses the fit unless the files or settings change (or `REFIT = True`).

In [ ]:
FIT_PARAMS = dict(pfa=1e-3, sigma=1.5, fraction_true=0.2, peak_wavelength=0.65, ROI_size=12, NA=1.49)

# Quick single-frame check of the spot detection before the full fit.
fig, axs = SupRes_F.example_spots_singleframe(
    DATA_DIR, pfa=FIT_PARAMS["pfa"], sigma=FIT_PARAMS["sigma"],
    fraction_true=FIT_PARAMS["fraction_true"], frame_index=10,
    variance=variance, read_noise=read_noise, rqe=rqe, offset_map=offset_map,
    gain_map=gain_map, peak_wavelength=FIT_PARAMS["peak_wavelength"],
)
plt.show()

In [ ]:
REFIT = False

fit_file = DATA_DIR / "Localisations.h5"
fit_record_file = DATA_DIR / "Localisations_fit_record.json"
fit_record = {"images": [p.name for p in downloaded["images"]], **FIT_PARAMS}

previous = json.loads(fit_record_file.read_text()) if fit_record_file.exists() else None
if REFIT or not fit_file.exists() or previous != fit_record:
    SupRes_F.fit_imaging_data(
        DATA_DIR, smoothing_function, gain_map, offset_map, rqe, read_noise,
        variance=variance, image_type=".tif", **FIT_PARAMS,
    )
    fit_record_file.write_text(json.dumps(fit_record, indent=1))
else:
    print(f"Reusing {fit_file} (same files and settings)")

---
## 4 — Quality-control filters

The cuts used for the paper, copied from the analysis notebook (sizes in nm, converted to camera px).

In [ ]:
loc_data = pd.read_hdf(fit_file)
print(f"Raw localisations: {len(loc_data):,}")

loc_data = loc_data[loc_data["xc_err"] < 30 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["yc_err"] < 30 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["s_x_err"] < 30 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["s_y_err"] < 30 / PIXEL_SIZE_NM]
loc_data = loc_data[(loc_data["s_x"] > 100 / PIXEL_SIZE_NM) & (loc_data["s_x"] < 175 / PIXEL_SIZE_NM)]
loc_data = loc_data[(loc_data["s_y"] > 100 / PIXEL_SIZE_NM) & (loc_data["s_y"] < 175 / PIXEL_SIZE_NM)]

loc_data = loc_data[loc_data["A_B_err"] < 0.01]
loc_data = loc_data[loc_data["A_G_err"] < 0.01]
loc_data = loc_data[loc_data["A_R_err"] < 0.01]

loc_data = loc_data[loc_data["A_B"] > 0.005]
loc_data = loc_data[loc_data["A_G"] > 0.05]
loc_data = loc_data[loc_data["A_R"] > 0.5]

loc_data = loc_data[loc_data["bg_B_err"] < 0.15]
loc_data = loc_data[loc_data["bg_G_err"] < 0.15]
loc_data = loc_data[loc_data["bg_R_err"] < 0.15]

loc_data = loc_data[~((loc_data["spot_snr"] < 2) & (loc_data["photons"] < 500)
                      & (loc_data["spot_matched_filter_response"] < 100))]
loc_data = loc_data[loc_data["spot_background_std"] > 10]
loc_data = loc_data[loc_data["chi_sqr"] < 1.5]
print(f"After QC: {len(loc_data):,}")

---
## 5 — Drift correction (AIM) and linking

Localisations of one molecule in consecutive frames (within the median localisation error,
up to 2 dark frames) are merged into one.

In [ ]:
drift_corrector = Drift_Correction_Functions()

info = [{
    "Width": width,
    "Height": height,
    "Frames": int(loc_data["frame"].max()),
    "Pixelsize": PIXEL_SIZE_NM,
}]

corrected_locs, drift_result = drift_corrector.undrift(
    locs=loc_data.to_records(index=False),
    info=info,
    method="aim",
    segmentation=20,
    intersect_d=20 / PIXEL_SIZE_NM,
    roi_r=60 / PIXEL_SIZE_NM,
)
corrected_locs = pd.DataFrame(corrected_locs)

plt.plot(drift_result.drift_x * PIXEL_SIZE_NM, label="x")
plt.plot(drift_result.drift_y * PIXEL_SIZE_NM, label="y")
plt.xlabel("frame"); plt.ylabel("drift / nm"); plt.legend()
plt.show()

# Frame count from the fitted data, so FAST_MODE (first file only) gets the right number.
n_frames = int(loc_data["frame"].max()) + 1
link_r = 0.5 * (np.median(corrected_locs["xc_err"]) + np.median(corrected_locs["yc_err"]))
linked = link_localisations(corrected_locs, n_frames=n_frames, r_max=link_r, max_dark_time=2)
print(f"Linked localisations: {len(linked):,}")

---
## 6 — Remove fiducials

Fiducial markers bind for most of the movie. Bright, dense regions of a 1x render that hold
localisations in at least 7.5% of frames are removed.

In [ ]:
info = [{"Width": width, "Height": height, "Frames": int(linked["frame"].max()), "Pixelsize": PIXEL_SIZE_NM}]
image_1x = render.render(locs=linked.to_records(index=False), info=info, blur_method="smooth", oversampling=1)[1]

region_centers, binary_mask, threshold, _ = drift_corrector.detect_high_density_regions_from_image(
    smoothed_image=image_1x,
    histogram_bins=1024,
    threshold_percentile=99.94,
    pixelsize=PIXEL_SIZE_NM,
    title="Fiducial detection",
)
selected_puncta, _ = drift_corrector.select_puncta_from_regions(
    locs=linked.to_records(index=False),
    region_centres=region_centers,
    binary_mask=binary_mask,
    pixelsize=PIXEL_SIZE_NM,
    selection_box_size_nm=350.0,
    min_localisations_per_region=int(0.075 * info[0]["Frames"]),
    title="Fiducial selection",
    create_plot=True,
)
linked_fremoved = pd.DataFrame(FD.remove_puncta_locs(linked.to_records(index=False), selected_puncta))
print(f"{len(linked) - len(linked_fremoved):,} fiducial localisations removed, {len(linked_fremoved):,} left")

IO.write_h5_database(df=linked_fremoved, filepath=DATA_DIR / "Localisations_Undrifted_Linked_FiducialsRemoved.h5",
                     normalise_photons=False)

---
## 7 — Separate ATTO 655 and Cy3B

A two-component Gaussian mixture on each localisation's (A_R, A_G) fingerprint; localisations
assigned with less than 95% confidence, or that are outliers to both, are left out. The
channel with the higher mean A_R is ATTO 655 (red-emitting), the other Cy3B.

In [ ]:
assigned_locs, unmix_info = SM_E.unmix_channels(
    linked_fremoved,
    n_channels=2,
    channels_to_use=["A_R", "A_G"],
    confidence_threshold=0.95,
    false_positive_rate=0.05,
    initial_guess_method="histogram_peaks",
    gmm_fit_method="fixed",
    covariance_type="full",
    outlier_rejection="mahalanobis",
    mestimator_type="tukey",
    initial_guess_percentile=50,
    initial_guess_scale=0.5,
    verbose=True,
    plot_results=True,
)

channel_ids = [c for c in sorted(assigned_locs["channel"].unique()) if c >= 0]
channel_mean_AR = {c: assigned_locs.loc[assigned_locs["channel"] == c, "A_R"].mean() for c in channel_ids}
ATTO655_ID = max(channel_mean_AR, key=channel_mean_AR.get)
CY3B_ID = min(channel_mean_AR, key=channel_mean_AR.get)
atto655 = assigned_locs[assigned_locs["channel"] == ATTO655_ID].reset_index(drop=True)
cy3b = assigned_locs[assigned_locs["channel"] == CY3B_ID].reset_index(drop=True)
print(f"ATTO 655 (channel {ATTO655_ID}, mean A_R {channel_mean_AR[ATTO655_ID]:.3f}): {len(atto655):,} localisations")
print(f"Cy3B     (channel {CY3B_ID}, mean A_R {channel_mean_AR[CY3B_ID]:.3f}): {len(cy3b):,} localisations")

---
## 8 — Figure 5b — two-colour render

ATTO 655 (vimentin) in `hot`, Cy3B (mitochondria) in cyan. White box: panel c; cyan box: panel d
(camera px).

In [ ]:
OVERSAMPLING = 8
SR_PX_NM = PIXEL_SIZE_NM / OVERSAMPLING
BOX_C = dict(x0=275, y0=325, w=150, h=150)
BOX_D = dict(x0=105, y0=525, w=160, h=int(160 / 2.12))

info = [{"Width": width, "Height": height, "Frames": int(linked_fremoved["frame"].max()), "Pixelsize": PIXEL_SIZE_NM}]
min_blur = 4 * PIXEL_SIZE_NM / 1000
_, img_atto655 = render.render(atto655.to_records(index=False), info, oversampling=OVERSAMPLING,
                               blur_method="gaussian", min_blur_width=min_blur)
_, img_cy3b = render.render(cy3b.to_records(index=False), info, oversampling=OVERSAMPLING,
                            blur_method="gaussian", min_blur_width=min_blur)

overlay_kw = dict(cmaps=["hot", "cyan"],
                  vmins=[np.percentile(img_atto655, 1), np.percentile(img_cy3b, 1)],
                  vmaxs=[np.percentile(img_atto655, 99.9), np.percentile(img_cy3b, 99.9)],
                  brightness_boost=[1, 1], pixelsize=SR_PX_NM)


def crop(img, box):
    return img[box["y0"] * OVERSAMPLING:(box["y0"] + box["h"]) * OVERSAMPLING,
               box["x0"] * OVERSAMPLING:(box["x0"] + box["w"]) * OVERSAMPLING]


fig, ax = plotter.one_column_plot(height=2, width=2)
ax = plotter.multichannel_overlay_plot(ax, [img_atto655, img_cy3b], scalebarsize=10000, scalebarlabel="10 μm",
                                       **overlay_kw)
for box, color in [(BOX_C, "white"), (BOX_D, "cyan")]:
    ax.add_patch(patches.Rectangle((box["x0"] * OVERSAMPLING, box["y0"] * OVERSAMPLING), box["w"] * OVERSAMPLING,
                                   box["h"] * OVERSAMPLING, linewidth=0.5, edgecolor=color, facecolor="none"))
plt.savefig(FIG_DIR / "Massive_Cells_Render_Box.svg", dpi=600, format="svg")
plt.show()

---
## 9 — Figure 5d — mitochondria zoom

In [ ]:
fig, ax = plotter.one_column_plot(height=2 / 2.12, width=2)
ax = plotter.multichannel_overlay_plot(ax, [crop(img_atto655, BOX_D), crop(img_cy3b, BOX_D)],
                                       scalebarsize=1000, scalebarlabel="1 μm", **overlay_kw)
plt.savefig(FIG_DIR / "Massive_Cells_Render_Zoomin_Mitochondria.svg", dpi=600, format="svg")
plt.show()

---
## 10 — Figure 5c — zoom and line profiles

Two lines drawn by hand on the published zoom (SR px within the crop, as (row, col) end points).
The intensity along each is binned (17.25 nm bins) and fitted with two Gaussians, giving the
peak separation and each peak's FWHM. `PROFILE_IMAGE` sets which channel is sampled: the analysis
used for the paper sampled the Cy3B render, while the caption describes vimentin (ATTO 655)
filaments; set it to `"ATTO 655"` to sample that channel instead.

In [ ]:
PROFILE_IMAGE = "Cy3B"   # as in the paper's analysis; "ATTO 655" samples the vimentin channel
LINES = [   # ((row0, col0), (row1, col1)) in camera px within BOX_C, then the x-limit of its plot (nm)
    (((4, 17), (12, 20)), 600),
    (((35, 87), (39, 97)), 300),
]

zoom_imgs = {"ATTO 655": crop(img_atto655, BOX_C), "Cy3B": crop(img_cy3b, BOX_C)}

fig, ax = plotter.one_column_plot(height=2, width=2)
ax = plotter.multichannel_overlay_plot(ax, [zoom_imgs["ATTO 655"], zoom_imgs["Cy3B"]],
                                       scalebarsize=1000, scalebarlabel="1 μm", **overlay_kw)
line_pixels = []
for ((r0, c0), (r1, c1)), _ in LINES:
    rows, cols = bresenham_line(r0 * OVERSAMPLING, c0 * OVERSAMPLING, r1 * OVERSAMPLING, c1 * OVERSAMPLING)
    line_pixels.append((rows, cols))
    ax.plot(cols, rows, color="white", linewidth=1.5, ls="-" if len(line_pixels) == 1 else "--")
plt.savefig(FIG_DIR / "Massive_Cells_Render_Zoomin.svg", dpi=600, format="svg")
plt.show()

In [ ]:
def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2))


def double_gaussian(x, A1, mu1, s1, A2, mu2, s2, offset):
    return gaussian(x, A1, mu1, s1) + gaussian(x, A2, mu2, s2) + offset


FWHM_PER_SIGMA = 2 * np.sqrt(2 * np.log(2))
BIN_NM = PIXEL_SIZE_NM / 4

fig, axs = plotter.one_column_plot(npanels=2, ratios=[1, 1], height=1.7, width=1)
for ax, (rows, cols), (_, xmax) in zip(axs, line_pixels, LINES):
    values = zoom_imgs[PROFILE_IMAGE][rows, cols]
    dist = np.concatenate([[0.0], np.cumsum(np.hypot(np.diff(rows.astype(float)), np.diff(cols.astype(float))))]) * SR_PX_NM
    bins = np.arange(dist[0], dist[-1] + BIN_NM, BIN_NM)
    sums, edges = np.histogram(dist, bins=bins, weights=values)
    counts, _ = np.histogram(dist, bins=bins)
    means = np.where(counts > 0, sums / np.maximum(counts, 1), 0.0)
    centres = (edges[:-1] + edges[1:]) / 2
    ax.bar(centres, means, width=BIN_NM * 0.75, color="#888", edgecolor="k", linewidth=0.4)

    peaks, props = find_peaks(means, distance=3, prominence=means.max() * 0.2)
    if len(peaks) >= 2:
        p1, p2 = peaks[np.argsort(props["prominences"])[-2:]]
        p0 = [means[p1], centres[p1], 2 * BIN_NM, means[p2], centres[p2], 2 * BIN_NM, 0.0]
        try:
            popt, pcov = curve_fit(double_gaussian, centres, means, p0=p0, bounds=(0, np.inf), maxfev=5000)
            perr = np.sqrt(np.diag(pcov))
            x_fit = np.linspace(centres[0], centres[-1], 500)
            for (a, mu, s), style in [(popt[0:3], "r--"), (popt[3:6], "g--")]:
                ax.plot(x_fit, gaussian(x_fit, a, mu, s) + popt[6], style, linewidth=0.75)
            print(f"peak separation {abs(popt[4] - popt[1]):.1f} +/- {np.hypot(perr[1], perr[4]):.1f} nm, "
                  f"FWHM {FWHM_PER_SIGMA * popt[2]:.0f} and {FWHM_PER_SIGMA * popt[5]:.0f} nm")
        except RuntimeError:
            print("double-Gaussian fit did not converge")
    else:
        print("fewer than two peaks along this line (expected in FAST_MODE)")
    ax.set_xlim([0, xmax])
    ax.set_yticklabels([])
    ax.grid(lw=0.5, alpha=0.5, ls="--", color="gray")
axs[1].set_xlabel("distance / nm", fontsize=8)
plt.savefig(FIG_DIR / "Massive_Cells_Cutthroughs.svg", dpi=600, format="svg")
plt.show()

---
## 11 — Figure 5e — FRC of each channel

Each channel's FRC is computed from its own localisations. (The published Cy3B curve was the
ATTO 655 one computed twice; the paper's 60 nm values are being re-checked.)

In [ ]:
def frc_curve(locs):
    """FIRE resolution and FRC curve of one set of localisations (camera px, with frame).

    Passing the frame column makes fire() split the two halves by time blocks, not row order.
    """
    xyf = locs[["xc", "yc", "frame"]].to_numpy(dtype=float)
    xyf[:, 0] -= xyf[:, 0].min()
    xyf[:, 1] -= xyf[:, 1].min()
    nx = int(xyf[:, 0].max()) + 1
    ny = int(xyf[:, 1].max()) + 1
    res, frc, res_hi, res_lo = fire(positions=xyf, nx=nx, ny=ny, zoom=FRC_ZOOM, n_blocks=50,
                                    pixel_size_nm=PIXEL_SIZE_NM)
    # Spatial frequency of each FRC bin in cycles/nm, kept below the SR-pixel Nyquist limit.
    q = np.arange(len(frc)) / (max(nx, ny) * FRC_ZOOM)   # cycles per SR pixel
    keep = (q > 0) & (q < 0.6)
    q_nm = q / (PIXEL_SIZE_NM / FRC_ZOOM)
    return dict(res=res, res_hi=res_hi, res_lo=res_lo, q_nm=q_nm[keep], frc=frc[keep])


def plot_frc(ax, curve, label, color, span_color):
    ax = plotter.line_plot(ax, curve["q_nm"], curve["frc"], label=label, color=color)
    ax.axvline(1 / curve["res"], color=span_color, ls="-", lw=1, label=f"FIRE = {curve['res']:.0f} nm")
    ax.axvspan(1 / curve["res_lo"], 1 / curve["res_hi"], alpha=0.15, color=span_color)
    return ax


FRC_ZOOM = 8   # SR pixels per camera pixel

In [ ]:
frc_atto655 = frc_curve(atto655)
frc_cy3b = frc_curve(cy3b)
for name, c in [("ATTO 655", frc_atto655), ("Cy3B", frc_cy3b)]:
    print(f"{name:8s}: FIRE {c['res']:.1f} nm (+/- {abs(c['res_hi'] - c['res_lo']) / 2:.1f} nm)")

fig, ax = plotter.one_column_plot(width=2.3, height=1.2)
ax = plot_frc(ax, frc_atto655, "ATTO 655", "darkred", "crimson")
ax = plot_frc(ax, frc_cy3b, "Cy3B", "darkgreen", "green")
ax.axhline(1 / 7, color="gray", ls="--", lw=1)
ax.set_xlabel("1 / spatial frequency (nm)", fontsize=8)
ax.set_ylabel("FRC", fontsize=8)
ax.set_ylim([0, 1.05])
ax.set_xscale("log")
tick_nm = [500, 100, 20]
ax.set_xticks([1 / d for d in tick_nm])
ax.set_xticklabels([f"{d} nm" for d in tick_nm])
ax.set_xlim(1 / 500, 1 / 15)
ax.legend(fontsize=6)
plt.tight_layout()
plt.savefig(FIG_DIR / "FRC_MassiveCells.svg", dpi=600, format="svg")
plt.show()